# Tugas 4 – Pemerolehan Informasi dan Penambangan Teks
## Search Engine dengan Peringkasan Teks Extractive (TF-IDF From Scratch)

**Mata Kuliah:** Pemerolehan Informasi dan Penambangan Teks  
**Dosen:** Putra Pandu Adikara, S.Kom., M.Kom.  
**Tahun:** 2026

---
### Anggota Kelompok:
| No | Nama | NIM |
|----|------|-----|
| 1  | Nicolas Gabriel Siahaan | 235150200111038 |
| 2  | Dzaky Rezandi | 235150207111006 |
| 3  | Rafly Januar Raharjo | 235150201111011 |
| 4  | M. Naufal Al Farizki | 235150207111032 |

---

### Deskripsi Tugas
Tugas ini merupakan modifikasi dari Tugas 3. Pada 3 hasil pencarian teratas, program menampilkan **ringkasan teks extractive** berupa **3 kalimat dengan skor TF-IDF tertinggi** (sebagai *excerpt*).  
Implementasi TF-IDF dan inverted index dibuat **from scratch** tanpa library TF-IDF seperti sklearn.

## 1. Import Library & Konfigurasi

In [2]:
!pip install Sastrawi

In [3]:
import re
import math
import json
from collections import defaultdict, Counter
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

CORPUS_FILE      = 'corpus.txt'
TOP_K            = 10        # Jumlah hasil pencarian
TOP_SUMMARIZE    = 3         # Jumlah dokumen teratas yang diringkas
TOP_SENTENCES    = 3         # Jumlah kalimat per ringkasan
MAX_PREVIEW_CHARS = 200

print('Library berhasil diimport!')
print(f'Konfigurasi: TOP_K={TOP_K}, TOP_SUMMARIZE={TOP_SUMMARIZE}, TOP_SENTENCES={TOP_SENTENCES}')

# Inisialisasi Sastrawi
print('Menginisialisasi Sastrawi (Stopword & Stemmer)...')
stopword_remover = StopWordRemoverFactory().create_stop_word_remover()
stemmer          = StemmerFactory().create_stemmer()
print('Siap!')

Library berhasil diimport!
Konfigurasi: TOP_K=10, TOP_SUMMARIZE=3, TOP_SENTENCES=3
Menginisialisasi Sastrawi (Stopword & Stemmer)...
Siap!


## 2. Parsing Korpus

In [4]:
def parse_corpus(filepath):
    """
    Membaca dan mem-parsing file corpus.txt dengan format XML sederhana.
    """
    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()

    documents = []
    doc_blocks = re.findall(r'<DOC>(.*?)</DOC>', content, re.DOTALL | re.IGNORECASE)

    for block in doc_blocks:
        doc_id = re.search(r'<ID>(.*?)</ID>',     block, re.DOTALL | re.IGNORECASE)
        nim    = re.search(r'<NIM>(.*?)</NIM>',   block, re.DOTALL | re.IGNORECASE)
        title  = re.search(r'<TITLE>(.*?)</TITLE>', block, re.DOTALL | re.IGNORECASE)
        url    = re.search(r'<URL>(.*?)</URL>',   block, re.DOTALL | re.IGNORECASE)
        text   = re.search(r'<TEXT>(.*?)</TEXT>', block, re.DOTALL | re.IGNORECASE)

        if doc_id and text:
            documents.append({
                'id'   : doc_id.group(1).strip(),
                'nim'  : nim.group(1).strip()   if nim   else '',
                'title': title.group(1).strip() if title else '',
                'url'  : url.group(1).strip()   if url   else '',
                'text' : text.group(1).strip()
            })

    return documents

documents = parse_corpus(CORPUS_FILE)
print(f'Total dokumen berhasil diparsing: {len(documents)}')

Total dokumen berhasil diparsing: 479


## 3. Preprocessing Teks

In [5]:
def preprocess(text):
    """
    Preprocessing teks bahasa Indonesia:
    1. Lowercase
    2. Hapus tanda baca
    3. Sastrawi: Stopword Removal
    4. Sastrawi: Stemming
    5. Tokenisasi
    """
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)  # Hapus tanda baca
    text = stopword_remover.remove(text)        # Hapus Stopword
    text = stemmer.stem(text)                   # Stemming

    tokens = text.split()
    tokens = [t for t in tokens if len(t) >= 3] # Hapus token pendek
    return tokens


def split_sentences(text):
    """
    Memecah teks menjadi daftar kalimat.
    Menggunakan tanda baca akhir kalimat: . ! ?
    Mengembalikan list of (index, kalimat_asli).
    """
    # Pisahkan berdasarkan . ! ? diikuti spasi atau akhir string
    raw_sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    sentences = []
    for idx, s in enumerate(raw_sentences):
        s = s.strip()
        if len(s) > 20:  # Abaikan kalimat terlalu pendek
            sentences.append((idx, s))
    return sentences


# Uji coba
sample_text = documents[0]['text']
sample_tokens = preprocess(sample_text)
sample_sentences = split_sentences(sample_text)
print(f'Teks asli (50 char pertama): {sample_text[:50]}')
print(f'Hasil preprocessing (10 token pertama): {sample_tokens[:10]}')
print(f'Jumlah kalimat pada dokumen pertama: {len(sample_sentences)}')

Teks asli (50 char pertama): Kadargula darah yang tiba-tiba melonjak perlu sege
Hasil preprocessing (10 token pertama): ['kadargula', 'darah', 'tiba', 'tiba', 'lonjak', 'perlu', 'segera', 'tangan', 'kembang', 'jadi']
Jumlah kalimat pada dokumen pertama: 26


## 4. Pembuatan Inverted Index

In [ ]:
def build_inverted_index(documents):
    """
    Membangun Inverted Index dari kumpulan dokumen.

    Struktur inverted index:
    {
        'term': {
            'doc_id_1': frekuensi_kemunculan,
            ...
        },
        ...
    }
    """
    inverted_index = defaultdict(dict)
    doc_tokens     = {}  # {doc_id: [tokens]}

    for doc in documents:
        doc_id    = doc['id']
        full_text = doc['title'] + ' ' + doc['text']
        tokens    = preprocess(full_text)

        doc_tokens[doc_id] = tokens

        term_freq = Counter(tokens)
        for term, freq in term_freq.items():
            inverted_index[term][doc_id] = freq

    return dict(inverted_index), doc_tokens


print('Membangun inverted index...')
inverted_index, doc_tokens = build_inverted_index(documents)

print(f'Jumlah unique terms dalam indeks: {len(inverted_index)}')
print(f'Jumlah dokumen terindeks        : {len(doc_tokens)}')

sample_terms = list(inverted_index.keys())[:3]
print('\nContoh inverted index:')
for term in sample_terms:
    postings = inverted_index[term]
    print(f'  "{term}" -> {dict(list(postings.items())[:3])} ...(total {len(postings)} dokumen)')

Membangun inverted index...


## 5. Perhitungan TF-IDF (From Scratch)

In [ ]:
def compute_tf(term, doc_tokens_list):
    """
    TF logarithmic:
    tf(t,d) = 1 + log10(count(t,d))  jika count > 0, else 0
    """
    count = doc_tokens_list.count(term)
    if count > 0:
        return 1 + math.log10(count)
    return 0.0


def compute_idf(term, inverted_index, total_docs):
    """
    IDF:
    idf(t) = log10(N / df(t))
    """
    df = len(inverted_index.get(term, {}))
    if df == 0:
        return 0.0
    return math.log10(total_docs / df)


def compute_tfidf(term, doc_tokens_list, inverted_index, total_docs):
    """TF-IDF = TF(t,d) x IDF(t)"""
    tf  = compute_tf(term, doc_tokens_list)
    idf = compute_idf(term, inverted_index, total_docs)
    return tf * idf


def build_tfidf_matrix(documents, doc_tokens, inverted_index):
    """
    Membangun matriks TF-IDF untuk semua dokumen (sparse dict of dict).
    """
    N = len(documents)
    tfidf_matrix = {}

    for doc in documents:
        doc_id       = doc['id']
        tokens       = doc_tokens[doc_id]
        unique_terms = set(tokens)
        tfidf_matrix[doc_id] = {}

        for term in unique_terms:
            score = compute_tfidf(term, tokens, inverted_index, N)
            tfidf_matrix[doc_id][term] = score

    return tfidf_matrix


print('Menghitung TF-IDF untuk semua dokumen...')
tfidf_matrix = build_tfidf_matrix(documents, doc_tokens, inverted_index)

# Contoh top-10 term
sample_doc    = documents[0]['id']
sample_scores = sorted(tfidf_matrix[sample_doc].items(), key=lambda x: x[1], reverse=True)[:10]
print(f'\nTop-10 term TF-IDF tertinggi pada dokumen "{sample_doc}":')
print(f'  Judul: {documents[0]["title"]}\n')
for rank, (term, score) in enumerate(sample_scores, 1):
    print(f'  {rank:2}. {term:<25} TF-IDF = {score:.4f}')

## 6. Vector Space Model – Cosine Similarity

In [ ]:
def compute_query_vector(query_tokens, inverted_index, total_docs):
    """
    Menghitung vektor TF-IDF untuk query.
    """
    query_vector = {}
    for term in set(query_tokens):
        score = compute_tfidf(term, query_tokens, inverted_index, total_docs)
        query_vector[term] = score
    return query_vector


def cosine_similarity(vec_a, vec_b):
    """
    Cosine Similarity:
    cos(q,d) = (q · d) / (|q| * |d|)
    """
    common_terms = set(vec_a.keys()) & set(vec_b.keys())
    dot_product  = sum(vec_a[t] * vec_b[t] for t in common_terms)

    if dot_product == 0:
        return 0.0

    mag_a = math.sqrt(sum(v**2 for v in vec_a.values()))
    mag_b = math.sqrt(sum(v**2 for v in vec_b.values()))

    if mag_a == 0 or mag_b == 0:
        return 0.0

    return dot_product / (mag_a * mag_b)


def search(query_str, documents, doc_tokens, tfidf_matrix, inverted_index, top_k=10):
    """
    Fungsi utama pencarian menggunakan Vector Space Model (TF-IDF + Cosine Similarity).
    """
    N = len(documents)

    query_tokens = preprocess(query_str)
    if not query_tokens:
        return []

    query_vector = compute_query_vector(query_tokens, inverted_index, N)

    # Candidate retrieval menggunakan inverted index
    candidate_docs = set()
    for term in query_tokens:
        if term in inverted_index:
            candidate_docs.update(inverted_index[term].keys())

    scores = []
    for doc_id in candidate_docs:
        doc_vector = tfidf_matrix.get(doc_id, {})
        sim = cosine_similarity(query_vector, doc_vector)
        if sim > 0:
            scores.append((doc_id, sim))

    scores.sort(key=lambda x: x[1], reverse=True)
    top_results = scores[:top_k]

    doc_map = {d['id']: d for d in documents}
    results = []
    for doc_id, sim in top_results:
        doc     = doc_map.get(doc_id, {})
        preview = doc.get('text', '')[:MAX_PREVIEW_CHARS]
        results.append({
            'rank'   : len(results) + 1,
            'doc_id' : doc_id,
            'score'  : sim,
            'title'  : doc.get('title', ''),
            'text'   : doc.get('text', ''),
            'preview': preview
        })

    return results


print('Fungsi search engine siap digunakan!')

## 7. Simpan Inverted Index & TF-IDF ke File (JSON)

In [ ]:
with open('inverted_index.json', 'w', encoding='utf-8') as f:
    json.dump(inverted_index, f, ensure_ascii=False, indent=2)

with open('tfidf_matrix.json', 'w', encoding='utf-8') as f:
    json.dump(tfidf_matrix, f, ensure_ascii=False, indent=2)

print('Inverted index disimpan ke: inverted_index.json')
print('TF-IDF matrix  disimpan ke: tfidf_matrix.json')
print(f'Ukuran inverted index : {len(inverted_index)} entri')
print(f'Ukuran TF-IDF matrix  : {len(tfidf_matrix)} dokumen')

## 8. Peringkasan Teks Extractive dengan TF-IDF (From Scratch)

### Konsep
Peringkasan teks *extractive* bekerja dengan cara memilih kalimat-kalimat **paling penting** dari teks asli tanpa mengubah kata-katanya.  
Skor kepentingan setiap kalimat dihitung berdasarkan **jumlah TF-IDF semua token dalam kalimat tersebut**.

### Rumus Skor Kalimat:
$$\text{score}(s) = \sum_{t \in s} \text{TF-IDF}(t, d)$$

dimana:
- $s$ = kalimat
- $t$ = token dalam kalimat $s$
- $d$ = dokumen yang memuat kalimat $s$

In [ ]:
def score_sentence(sentence_text, doc_tfidf_vector):
    """
    Menghitung skor TF-IDF sebuah kalimat dari dokumen.

    Skor kalimat = rata-rata TF-IDF semua token dalam kalimat.
    (Rata-rata digunakan agar kalimat panjang tidak selalu menang.)

    Parameter:
        sentence_text    : string kalimat asli
        doc_tfidf_vector : dict {term: tfidf_score} dari dokumen

    Return:
        float skor kalimat
    """
    tokens = preprocess(sentence_text)
    if not tokens:
        return 0.0

    total_score = sum(doc_tfidf_vector.get(t, 0.0) for t in tokens)
    # Rata-rata agar adil antara kalimat pendek dan panjang
    return total_score / len(tokens)


def extractive_summarize(doc_text, doc_tfidf_vector, top_n=3):
    """
    Melakukan peringkasan teks extractive dengan TF-IDF.

    Algoritma:
    1. Pecah teks menjadi kalimat
    2. Hitung skor TF-IDF setiap kalimat
    3. Pilih top_n kalimat dengan skor tertinggi
    4. Kembalikan dalam urutan kemunculan asli (bukan urutan skor)

    Parameter:
        doc_text         : teks dokumen asli (string)
        doc_tfidf_vector : dict {term: tfidf_score} dokumen
        top_n            : jumlah kalimat yang dipilih

    Return:
        list of dict: [{'index': idx, 'sentence': str, 'score': float}, ...]
    """
    sentences = split_sentences(doc_text)

    if not sentences:
        return []

    # Beri skor setiap kalimat
    scored = []
    for idx, sent in sentences:
        score = score_sentence(sent, doc_tfidf_vector)
        scored.append({'index': idx, 'sentence': sent, 'score': score})

    # Pilih top_n kalimat berdasarkan skor tertinggi
    top_sentences = sorted(scored, key=lambda x: x['score'], reverse=True)[:top_n]

    # Kembalikan dalam urutan asli teks (bukan urutan skor)
    top_sentences.sort(key=lambda x: x['index'])

    return top_sentences


print('Fungsi peringkasan teks extractive siap digunakan!')

# --- Uji coba pada dokumen pertama ---
test_doc       = documents[0]
test_tfidf_vec = tfidf_matrix[test_doc['id']]
test_summary   = extractive_summarize(test_doc['text'], test_tfidf_vec, top_n=3)

print(f'\n[UJI COBA] Dokumen: {test_doc["title"]}')
print(f'ID: {test_doc["id"]}')
print('--- Ringkasan (3 kalimat terpenting) ---')
for item in test_summary:
    print(f'  [{item["index"]}] (skor={item["score"]:.4f}) {item["sentence"]}')

## 9. Fungsi Display Hasil Pencarian + Ringkasan

In [ ]:
def display_results_with_summary(query_str, results, tfidf_matrix,
                                  top_summarize=3, top_sentences=3):
    """
    Menampilkan hasil pencarian.
    - Untuk top_summarize hasil teratas: tampilkan ringkasan extractive (TF-IDF)
    - Untuk sisa hasil: tampilkan preview singkat

    Parameter:
        query_str      : string kueri
        results        : list hasil dari fungsi search()
        tfidf_matrix   : dict {doc_id: {term: skor}}
        top_summarize  : berapa hasil teratas yang diringkas (default 3)
        top_sentences  : berapa kalimat per ringkasan (default 3)
    """
    print(f'\n{"="*70}')
    print(f'HASIL PENCARIAN untuk kueri: "{query_str}"')
    print(f'Jumlah hasil ditemukan: {len(results)}')
    print(f'{"="*70}\n')

    if not results:
        print('Tidak ada dokumen yang relevan ditemukan.')
        return

    for r in results:
        rank   = r['rank']
        doc_id = r['doc_id']
        title  = r['title']
        score  = r['score']
        text   = r['text']

        print(f'{rank}. {title} | ID={doc_id}')
        print(f'   Skor Cosine Similarity: {score:.6f}')

        if rank <= top_summarize:
            # Tampilkan ringkasan extractive TF-IDF
            doc_tfidf_vec = tfidf_matrix.get(doc_id, {})
            summary       = extractive_summarize(text, doc_tfidf_vec, top_n=top_sentences)

            print(f'   [RINGKASAN EXTRACTIVE – {top_sentences} kalimat TF-IDF tertinggi]')
            if summary:
                for item in summary:
                    idx  = item['index']
                    sent = item['sentence']
                    sc   = item['score']
                    print(f'   [{idx}] (skor={sc:.4f}) {sent}')
            else:
                print('   (Teks terlalu pendek untuk diringkas)')
        else:
            # Preview singkat untuk hasil ke-4 dan seterusnya
            preview = text[:MAX_PREVIEW_CHARS]
            print(f'   {preview}...')

        print()


print('Fungsi display hasil + ringkasan siap digunakan!')

## 10. Uji Coba Pencarian dengan Ringkasan

### Query 1 & 2 – Mahasiswa 1 (Nicolas Gabriel Siahaan)

In [ ]:
query_mhs1_q1 = "gula darah diabetes insulin"

results_mhs1_q1 = search(query_mhs1_q1, documents, doc_tokens,
                          tfidf_matrix, inverted_index, top_k=TOP_K)
display_results_with_summary(query_mhs1_q1, results_mhs1_q1, tfidf_matrix,
                              top_summarize=TOP_SUMMARIZE, top_sentences=TOP_SENTENCES)

In [ ]:
query_mhs1_q2 = "vaksin campak imunisasi anak"

results_mhs1_q2 = search(query_mhs1_q2, documents, doc_tokens,
                          tfidf_matrix, inverted_index, top_k=TOP_K)
display_results_with_summary(query_mhs1_q2, results_mhs1_q2, tfidf_matrix,
                              top_summarize=TOP_SUMMARIZE, top_sentences=TOP_SENTENCES)

### Query 3 & 4 – Mahasiswa 2 (Dzaky Rezandi)

In [ ]:
query_mhs2_q1 = "protein karbohidrat gizi sahur puasa"

results_mhs2_q1 = search(query_mhs2_q1, documents, doc_tokens,
                          tfidf_matrix, inverted_index, top_k=TOP_K)
display_results_with_summary(query_mhs2_q1, results_mhs2_q1, tfidf_matrix,
                              top_summarize=TOP_SUMMARIZE, top_sentences=TOP_SENTENCES)

In [ ]:
# Ganti query sesuai query Tugas 3 milik Mahasiswa 2
query_mhs2_q2 = "stunting gizi buruk anak balita"

results_mhs2_q2 = search(query_mhs2_q2, documents, doc_tokens,
                          tfidf_matrix, inverted_index, top_k=TOP_K)
display_results_with_summary(query_mhs2_q2, results_mhs2_q2, tfidf_matrix,
                              top_summarize=TOP_SUMMARIZE, top_sentences=TOP_SENTENCES)

### Query 5 & 6 – Mahasiswa 3 (Rafly Januar Raharjo)

In [ ]:
# Ganti query sesuai query Tugas 3 milik Mahasiswa 3
query_mhs3_q1 = "kanker pengobatan kemoterapi tumor"

results_mhs3_q1 = search(query_mhs3_q1, documents, doc_tokens,
                          tfidf_matrix, inverted_index, top_k=TOP_K)
display_results_with_summary(query_mhs3_q1, results_mhs3_q1, tfidf_matrix,
                              top_summarize=TOP_SUMMARIZE, top_sentences=TOP_SENTENCES)

In [ ]:
# Ganti query sesuai query Tugas 3 milik Mahasiswa 3
query_mhs3_q2 = "tekanan darah hipertensi obat"

results_mhs3_q2 = search(query_mhs3_q2, documents, doc_tokens,
                          tfidf_matrix, inverted_index, top_k=TOP_K)
display_results_with_summary(query_mhs3_q2, results_mhs3_q2, tfidf_matrix,
                              top_summarize=TOP_SUMMARIZE, top_sentences=TOP_SENTENCES)

### Query 7 & 8 – Mahasiswa 4 (M. Naufal Al Farizki)

In [ ]:
# Ganti query sesuai query Tugas 3 milik Mahasiswa 4
query_mhs4_q1 = "kolesterol jantung lemak darah"

results_mhs4_q1 = search(query_mhs4_q1, documents, doc_tokens,
                          tfidf_matrix, inverted_index, top_k=TOP_K)
display_results_with_summary(query_mhs4_q1, results_mhs4_q1, tfidf_matrix,
                              top_summarize=TOP_SUMMARIZE, top_sentences=TOP_SENTENCES)

In [ ]:
# Ganti query sesuai query Tugas 3 milik Mahasiswa 4
query_mhs4_q2 = "vitamin suplemen daya tahan tubuh"

results_mhs4_q2 = search(query_mhs4_q2, documents, doc_tokens,
                          tfidf_matrix, inverted_index, top_k=TOP_K)
display_results_with_summary(query_mhs4_q2, results_mhs4_q2, tfidf_matrix,
                              top_summarize=TOP_SUMMARIZE, top_sentences=TOP_SENTENCES)

## 11. Pencarian Bebas / Interaktif (dengan Ringkasan)

In [ ]:
kueri_bebas = input('Masukkan kueri: ')
hasil = search(kueri_bebas, documents, doc_tokens,
               tfidf_matrix, inverted_index, top_k=TOP_K)
display_results_with_summary(kueri_bebas, hasil, tfidf_matrix,
                              top_summarize=TOP_SUMMARIZE, top_sentences=TOP_SENTENCES)

---
## Ringkasan Sistem

| Komponen | Keterangan |
|----------|------------|
| **Korpus** | corpus.txt (479 dokumen artikel kesehatan) |
| **Preprocessing** | Lowercase, hapus tanda baca, Sastrawi stopword + stemming |
| **Pembobotan** | TF-IDF from scratch (logarithmic TF × log IDF) |
| **Struktur Data** | Inverted Index (term → {doc_id: tf}) |
| **Model Retrieval** | Vector Space Model + Cosine Similarity |
| **Peringkasan** | Extractive summarization: skor kalimat = rata-rata TF-IDF token, pilih 3 kalimat tertinggi, tampilkan urutan asli |
| **Output** | inverted_index.json, tfidf_matrix.json |

### Alur Peringkasan Teks Extractive:
```
Teks Dokumen
     │
     ▼
split_sentences()  → [(idx, kalimat), ...]
     │
     ▼
score_sentence()   → skor = rata-rata TF-IDF token dalam kalimat
     │
     ▼
sort by score DESC → pilih Top-3 kalimat
     │
     ▼
sort by index ASC  → kembalikan ke urutan asli teks
     │
     ▼
Tampilkan sebagai excerpt [IDK]
```

In [ ]:
import json

# Setelah baris: tfidf_matrix = build_tfidf_matrix(documents, doc_tokens, inverted_index)

with open("tfidf.json", "w", encoding="utf-8") as f:
    json.dump(tfidf_matrix, f, ensure_ascii=False, indent=2)

with open("inverted_index.json", "w", encoding="utf-8") as f:
    json.dump(inverted_index, f, ensure_ascii=False, indent=2)

print("File berhasil disimpan!")